In [1]:
import pandas as pd

### Block 1: merge on a key
- `pd.merge(left, right, on="key")` or `left.merge(right, on="key")`
- the key is the column that appears in both tables
- default is an inner merge: only rows whose key is in BOTH tables stay
- rows with no match on the other side disappear (no error, no warning)
- one customer with 2 orders gives 2 result rows (details repeat)
- after merge, use groupby on columns from both tables

Watch out:
- the key name must exist in both tables, else KeyError
- always check `shape` before and after merge; a smaller result means rows were dropped
- save the result: `m = ...merge(...)`, merge does not change the original tables

In [2]:
import pandas as pd

customers = pd.DataFrame({"cust_id": [1, 2, 3, 4],
                          "name":    ["Asha", "Ravi", "Meena", "Dev"],
                          "city":    ["Indore", "Pune", "Delhi", "Pune"]})

orders = pd.DataFrame({"order_id": [11, 12, 13, 14, 15],
                       "cust_id":  [1, 2, 1, 3, 9],
                       "amount":   [500, 700, 300, 900, 250]})

print(customers)
print(orders)

   cust_id   name    city
0        1   Asha  Indore
1        2   Ravi    Pune
2        3  Meena   Delhi
3        4    Dev    Pune
   order_id  cust_id  amount
0        11        1     500
1        12        2     700
2        13        1     300
3        14        3     900
4        15        9     250


In [5]:
m = pd.merge(orders,customers,on="cust_id")
print(m)
print(orders.shape, customers.shape, m.shape)

   order_id  cust_id  amount   name    city
0        11        1     500   Asha  Indore
1        12        2     700   Ravi    Pune
2        13        1     300   Asha  Indore
3        14        3     900  Meena   Delhi
(5, 3) (4, 3) (4, 5)


In [6]:
m2 = orders.merge(customers, on="cust_id")
print(m2.equals(m))

True


In [7]:
print(m.groupby("name")["amount"].sum())
print(m.groupby("city")["amount"].sum())


name
Asha     800
Meena    900
Ravi     700
Name: amount, dtype: int64
city
Delhi     900
Indore    800
Pune      700
Name: amount, dtype: int64


### Block 2: how= (keeping unmatched rows)
- `how="inner"` (default): only keys in both tables
- `how="left"`: all rows of the left table, NaN where the right has no match
- `how="right"`: all rows of the right table
- `how="outer"`: all rows of both
- `indicator=True` adds `_merge` (both / left_only / right_only) to see what matched
- unmatched side gets NaN, so int columns become float
- customers who never ordered: `merge(customers, orders, how="left")`, then keep rows where an order column `isna()`
- handle the NaN afterwards with `fillna` / `dropna`

Watch out:
- "left" means the table written FIRST in merge; swapping the tables changes the result
- inner merge silently drops rows; check `shape`
- how must be one of inner, left, right, outer, else ValueError

In [8]:
inner = pd.merge(orders, customers, on="cust_id")
print(inner.shape)

(4, 5)


In [9]:
left = pd.merge(orders, customers, on="cust_id", how="left")
print(left)
print(left.shape) 

   order_id  cust_id  amount   name    city
0        11        1     500   Asha  Indore
1        12        2     700   Ravi    Pune
2        13        1     300   Asha  Indore
3        14        3     900  Meena   Delhi
4        15        9     250    NaN     NaN
(5, 5)


In [10]:
right = pd.merge(orders, customers, on="cust_id", how="right")
print(right)


   order_id  cust_id  amount   name    city
0      11.0        1   500.0   Asha  Indore
1      13.0        1   300.0   Asha  Indore
2      12.0        2   700.0   Ravi    Pune
3      14.0        3   900.0  Meena   Delhi
4       NaN        4     NaN    Dev    Pune


In [11]:
# outer: keep everything from both
outer = pd.merge(orders, customers, on="cust_id", how="outer")
print(outer)


   order_id  cust_id  amount   name    city
0      11.0        1   500.0   Asha  Indore
1      13.0        1   300.0   Asha  Indore
2      12.0        2   700.0   Ravi    Pune
3      14.0        3   900.0  Meena   Delhi
4       NaN        4     NaN    Dev    Pune
5      15.0        9   250.0    NaN     NaN


In [12]:
# indicator: adds a column _merge that says where each row came from
chk = pd.merge(orders, customers, on="cust_id", how="outer", indicator=True)
print(chk[["order_id", "cust_id", "name", "_merge"]])

   order_id  cust_id   name      _merge
0      11.0        1   Asha        both
1      13.0        1   Asha        both
2      12.0        2   Ravi        both
3      14.0        3  Meena        both
4       NaN        4    Dev  right_only
5      15.0        9    NaN   left_only


### Block 3: key names, same names, duplicates
- different key names: `merge(a, b, left_on="cid", right_on="cust_id")`; both key columns stay, drop one
- same name in a non-key column: gets `_x` and `_y`; choose with `suffixes=("_home","_work")`, or select only needed columns of the right table first
- duplicate keys in one table copy rows in the other: no error, totals become wrong
- check: `df["key"].is_unique`, and compare `shape` / `sum` before and after
- `validate="m:1"` raises MergeError if the right table has duplicate keys
- fix duplicates with `drop_duplicates("key")` once you know which row to keep

Watch out:
- `on=` needs the same name in both tables, else KeyError
- a silent wrong total is worse than an error; always check after merge
- `_x` / `_y` columns mean you forgot to handle a same-name column

In [13]:
import pandas as pd

# 1. key has a DIFFERENT name in each table
orders = pd.DataFrame({"order_id": [11, 12, 13],
                       "cid":      [1, 2, 3],
                       "amount":   [500, 700, 300]})
people = pd.DataFrame({"cust_id": [1, 2, 3],
                       "name":    ["Asha", "Ravi", "Meena"]})

m = pd.merge(orders, people, left_on="cid", right_on="cust_id")   # left key, right key
print(m)                                   # both key columns stay
m = m.drop(columns="cust_id")              # drop the extra one
print(m.columns.tolist())

# 2. a NON-key column has the same name in both tables
a = pd.DataFrame({"id": [1, 2], "city": ["Indore", "Pune"], "score": [80, 90]})
b = pd.DataFrame({"id": [1, 2], "city": ["Bhopal", "Delhi"], "bonus": [5, 6]})
print(pd.merge(a, b, on="id"))                                    # city_x, city_y
print(pd.merge(a, b, on="id", suffixes=("_home", "_work")))       # your own endings
print(pd.merge(a, b[["id", "bonus"]], on="id"))                   # or bring only what you need

# 3. DUPLICATE keys in the right table (the dangerous one)
c = pd.DataFrame({"cust_id": [1, 1, 2], "name": ["Asha", "Asha2", "Ravi"]})
o = pd.DataFrame({"order_id": [11, 12], "cust_id": [1, 2], "amount": [500, 700]})
bad = pd.merge(o, c, on="cust_id")
print(bad)                                 # order 11 appears twice
print(o["amount"].sum(), bad["amount"].sum())     # 1200 vs 1700, total is now wrong!

print(c["cust_id"].is_unique)              # False -> warning sign
# validate="m:1" means: many rows on the left, ONE row per key on the right
# pd.merge(o, c, on="cust_id", validate="m:1")     # MergeError because c has duplicate keys
fixed = pd.merge(o, c.drop_duplicates("cust_id"), on="cust_id", validate="m:1")
print(fixed)

   order_id  cid  amount  cust_id   name
0        11    1     500        1   Asha
1        12    2     700        2   Ravi
2        13    3     300        3  Meena
['order_id', 'cid', 'amount', 'name']
   id  city_x  score  city_y  bonus
0   1  Indore     80  Bhopal      5
1   2    Pune     90   Delhi      6
   id city_home  score city_work  bonus
0   1    Indore     80    Bhopal      5
1   2      Pune     90     Delhi      6
   id    city  score  bonus
0   1  Indore     80      5
1   2    Pune     90      6
   order_id  cust_id  amount   name
0        11        1     500   Asha
1        11        1     500  Asha2
2        12        2     700   Ravi
1200 1700
False
   order_id  cust_id  amount  name
0        11        1     500  Asha
1        12        2     700  Ravi
